In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [9]:
import pandas as pd 
import numpy as np  
df=pd.read_csv("train.csv")
df_test=pd.read_csv("test.csv")

In [10]:
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [11]:
len(df.columns)

16

In [12]:
import matplotlib.pyplot as py
import seaborn as sns

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [14]:
df["Year"].min(),df["Year"].max()

(np.int64(2022), np.int64(2025))

In [15]:
df["Year"].astype(int)-2022  #year between 0-3 

0         0
1         3
2         0
3         1
4         0
         ..
439135    1
439136    1
439137    1
439138    1
439139    1
Name: Year, Length: 439140, dtype: int64

In [16]:
df["Compound"].unique()


array(['HARD', 'MEDIUM', 'INTERMEDIATE', 'SOFT', 'WET'], dtype=object)

In [17]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['Compound'] = encoder.fit_transform(df['Compound'])

In [18]:
df["Compound"].unique
# to know mapped classes
encoder.classes_
#'HARD'-0, 'INTERMEDIATE'-1, 'MEDIUM'-2, 'SOFT'-3, 'WET'-4

array(['HARD', 'INTERMEDIATE', 'MEDIUM', 'SOFT', 'WET'], dtype=object)

In [19]:
df["Race"].str.replace("Grand Prix","").unique()
encoder=LabelEncoder()

In [20]:
df["Race"]=encoder.fit_transform(df["Race"])
df["Race"].unique().max()

np.int64(25)

In [21]:
len(df["Driver"].unique())

887

In [ ]:
x_train=df[[column for column in df.columns if column != 'PitNextLap' and  column != 'Driver']]
y_train=df['PitNextLap']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x_train,y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)  


In [33]:
print(x_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 351312 entries, 356074 to 121958
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      351312 non-null  int64  
 1   Compound                351312 non-null  int64  
 2   Race                    351312 non-null  int64  
 3   Year                    351312 non-null  int64  
 4   PitStop                 351312 non-null  int64  
 5   LapNumber               351312 non-null  int64  
 6   Stint                   351312 non-null  int64  
 7   TyreLife                351312 non-null  float64
 8   Position                351312 non-null  int64  
 9   LapTime (s)             351312 non-null  float64
 10  LapTime_Delta           351312 non-null  float64
 11  Cumulative_Degradation  351312 non-null  float64
 12  RaceProgress            351312 non-null  float64
 13  Position_Change         351312 non-null  float64
dtypes: float64(6), int64

In [ ]:
#Binary Classification
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
epochs=1000
learning_rate=0.01
mlp_clf=MLPClassifier(hidden_layer_sizes=[200,200,200],solver='adam',verbose=True,early_stopping=True,batch_size=1024,learning_rate='adaptive',random_state=42)
pipeline=make_pipeline(MinMaxScaler(),mlp_clf)
pipeline.fit(x_train,y_train)
y_proba = pipeline.predict_proba(
    x_test
)[:,1]
y_pred=(y_proba>0.3).astype(int)


Iteration 1, loss = 0.36418987
Validation score: 0.862063
Iteration 2, loss = 0.28558723
Validation score: 0.877775
Iteration 3, loss = 0.27683204
Validation score: 0.878715
Iteration 4, loss = 0.27195030
Validation score: 0.878715
Iteration 5, loss = 0.26839413
Validation score: 0.881504
Iteration 6, loss = 0.26602724
Validation score: 0.880166
Iteration 7, loss = 0.26454115
Validation score: 0.881931
Iteration 8, loss = 0.26379808
Validation score: 0.882130
Iteration 9, loss = 0.26244187
Validation score: 0.883269
Iteration 10, loss = 0.26050284
Validation score: 0.881049
Iteration 11, loss = 0.26003078
Validation score: 0.883553
Iteration 12, loss = 0.25967228
Validation score: 0.880052
Iteration 13, loss = 0.25854281
Validation score: 0.884436
Iteration 14, loss = 0.25795818
Validation score: 0.883212
Iteration 15, loss = 0.25692066
Validation score: 0.883468
Iteration 16, loss = 0.25663764
Validation score: 0.883269
Iteration 17, loss = 0.25550338
Validation score: 0.885034
Iterat

C:\Users\bishe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

In [43]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    f1_score
)
accuracy=accuracy_score(y_test,y_pred) # 0.888108
print(accuracy)
print(classification_report(y_test, y_pred))

0.8753244978822243
              precision    recall  f1-score   support

         0.0       0.96      0.88      0.92     70286
         1.0       0.64      0.86      0.73     17542

    accuracy                           0.88     87828
   macro avg       0.80      0.87      0.83     87828
weighted avg       0.90      0.88      0.88     87828



In [44]:
#train the model on the whole data
x=df[[column for column in df.columns if column != 'PitNextLap' and  column != 'Driver']]
y=df['PitNextLap']
pipeline.fit(x,y)


df_test["Year"].astype(int)-2022
df_test['Compound'] = encoder.fit_transform(df_test['Compound'])
df_test["Race"]=encoder.fit_transform(df_test["Race"])
df_test.info()

Iteration 1, loss = 0.34884718
Validation score: 0.877032
Iteration 2, loss = 0.28238945
Validation score: 0.879697
Iteration 3, loss = 0.27435157
Validation score: 0.882725
Iteration 4, loss = 0.26978215
Validation score: 0.884775
Iteration 5, loss = 0.26661720
Validation score: 0.883796
Iteration 6, loss = 0.26360089
Validation score: 0.886847
Iteration 7, loss = 0.26178792
Validation score: 0.886437
Iteration 8, loss = 0.26085744
Validation score: 0.886027
Iteration 9, loss = 0.25897450
Validation score: 0.886437
Iteration 10, loss = 0.25824482
Validation score: 0.887280
Iteration 11, loss = 0.25699057
Validation score: 0.886073
Iteration 12, loss = 0.25636046
Validation score: 0.888487
Iteration 13, loss = 0.25552386
Validation score: 0.887325
Iteration 14, loss = 0.25493315
Validation score: 0.888145
Iteration 15, loss = 0.25399747
Validation score: 0.888487
Iteration 16, loss = 0.25301911
Validation score: 0.889261
Iteration 17, loss = 0.25232338
Validation score: 0.889693
Iterat

In [45]:
x_test_final=df_test[
    [
        column for column in df.columns
        if column not in ['Driver', 'PitNextLap']
    ]
]
x_test_final.head()

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,2,6,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,2,0,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,2,6,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,3,24,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,0,25,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [46]:
df_test.shape
x_test_final.shape

(188165, 14)

In [47]:

y_pred_final = pipeline.predict(x_test_final)

submission = pd.DataFrame({
    'id': x_test_final['id'],
    'PitNextStop': y_pred_final.astype(int)
})

submission.to_csv('submission.csv', index=False)
print(submission.head())

       id  PitNextStop
0  439140            0
1  439141            0
2  439142            0
3  439143            0
4  439144            1


In [48]:
(y_pred_final==1).sum()

np.int64(35936)

In [51]:
x_train_xgboost=df[[column for column in df.columns if column != 'PitNextLap' and  column != 'Driver']]
y_train_xgboost=df['PitNextLap']
from sklearn.model_selection import train_test_split
x_train_xgboost,x_test_xgboost,y_train_xgboost,y_test_xgboost=train_test_split(
    x_train_xgboost,y_train_xgboost,
    test_size=0.2,
    random_state=42,
    stratify=y_train_xgboost
)  

In [52]:
x_train_xgboost.info()

<class 'pandas.core.frame.DataFrame'>
Index: 351312 entries, 152811 to 96127
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      351312 non-null  int64  
 1   Compound                351312 non-null  int64  
 2   Race                    351312 non-null  int64  
 3   Year                    351312 non-null  int64  
 4   PitStop                 351312 non-null  int64  
 5   LapNumber               351312 non-null  int64  
 6   Stint                   351312 non-null  int64  
 7   TyreLife                351312 non-null  float64
 8   Position                351312 non-null  int64  
 9   LapTime (s)             351312 non-null  float64
 10  LapTime_Delta           351312 non-null  float64
 11  Cumulative_Degradation  351312 non-null  float64
 12  RaceProgress            351312 non-null  float64
 13  Position_Change         351312 non-null  float64
dtypes: float64(6), int64(

In [62]:
#Applying Hyperparameter Tuning to XGBoost
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV


# Handle class imbalance
scale_pos_weight = (
    (y_train_xgboost == 0).sum()
    /
    (y_train_xgboost == 1).sum()
)

# Base XGBoost model
xgb_clf = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

# Hyperparameter search space
param_dist = {

    'n_estimators': [200, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# Randomized Search
random_search = RandomizedSearchCV(

    estimator=xgb_clf,

    param_distributions=param_dist,

    n_iter=15,

    scoring='roc_auc',

    cv=3,

    verbose=2,

    random_state=42,

    n_jobs=-1
)

# Train
random_search.fit(
    x_train_xgboost,
    y_train_xgboost
)
random_search.best_params_ #taking best parameters


Fitting 3 folds for each of 15 candidates, totalling 45 fits


{'subsample': 1.0,
 'n_estimators': 500,
 'min_child_weight': 5,
 'max_depth': 8,
 'learning_rate': 0.05,
 'gamma': 0.1,
 'colsample_bytree': 0.8}

In [65]:
#Applying XgBoost along with optimizing hyperparameters
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    subsample=1.0,
    n_estimators= 500,
    min_child_weight= 5,
 max_depth= 8,
 learning_rate= 0.05,
 gamma= 0.1,
 colsample_bytree=0.8
)
#hyperparameter tuning using cv

from sklearn.metrics import (
    accuracy_score, 
    classification_report,)
xgb_clf.fit(x_train_xgboost,y_train_xgboost,eval_set=[(x_test_xgboost,y_test_xgboost)],verbose=True)
y_pred_xgb=xgb_clf.predict(x_test_xgboost)
accuracy_xgb=accuracy_score(y_test_xgboost,y_pred_xgb)
print(accuracy_xgb) 
print(classification_report(y_test_xgboost,y_pred_xgb))

[0]	validation_0-auc:0.88651
[1]	validation_0-auc:0.92165
[2]	validation_0-auc:0.92151
[3]	validation_0-auc:0.92335
[4]	validation_0-auc:0.92668
[5]	validation_0-auc:0.92806
[6]	validation_0-auc:0.92889
[7]	validation_0-auc:0.92913
[8]	validation_0-auc:0.93004
[9]	validation_0-auc:0.93078
[10]	validation_0-auc:0.93105
[11]	validation_0-auc:0.93168
[12]	validation_0-auc:0.93190
[13]	validation_0-auc:0.93201
[14]	validation_0-auc:0.93235
[15]	validation_0-auc:0.93257
[16]	validation_0-auc:0.93293
[17]	validation_0-auc:0.93344
[18]	validation_0-auc:0.93373
[19]	validation_0-auc:0.93390
[20]	validation_0-auc:0.93410
[21]	validation_0-auc:0.93430
[22]	validation_0-auc:0.93444
[23]	validation_0-auc:0.93456
[24]	validation_0-auc:0.93473
[25]	validation_0-auc:0.93502
[26]	validation_0-auc:0.93523
[27]	validation_0-auc:0.93554
[28]	validation_0-auc:0.93573
[29]	validation_0-auc:0.93589
[30]	validation_0-auc:0.93612
[31]	validation_0-auc:0.93624
[32]	validation_0-auc:0.93656
[33]	validation_0-au

In [60]:
scale_pos_weight = (
    (y_train_xgboost == 0).sum()
    /
    (y_train_xgboost == 1).sum()
)
print(scale_pos_weight)

4.025563264430298


In [67]:
y_pred_final = xgb_clf.predict(x_test_final)

submission = pd.DataFrame({
    'id': x_test_final['id'],
    'PitNextStop': y_pred_final.astype(int)
})

submission.to_csv('submission2.csv', index=False)
print(submission.head())

       id  PitNextStop
0  439140            0
1  439141            0
2  439142            0
3  439143            0
4  439144            1
